In [ ]:
# Uninstall the pip version first
!pip uninstall lang-sam -y

# Clone it
!git clone https://github.com/luca-medeiros/lang-segment-anything.git

%cd /kaggle/working/lang-segment-anything
!pip install -e .

# Install in editable mode so you can edit source directly


In [ ]:
!python -c "from lang_sam import LangSAM; print('OK')"

In [ ]:
!python -c "import lang_sam; print(lang_sam.__file__)"

In [ ]:
!find /kaggle/working/lang-segment-anything -type f -name "*.py" | head -30


In [ ]:
!pip show lang-sam

In [ ]:
!cat /kaggle/working/lang-segment-anything/lang_sam/models/gdino.py

In [ ]:
lang_sam_path = "/kaggle/working/lang-segment-anything/lang_sam/lang_sam.py"

new_content = '''import numpy as np
from PIL import Image

from lang_sam.models.gdino import GDINO
from lang_sam.models.sam import SAM
from lang_sam.models.utils import DEVICE


class LangSAM:
    def __init__(self, sam_type="sam2.1_hiera_small", sam_ckpt_path: str | None = None, gdino_model_ckpt_path: str | None = None, gdino_processor_ckpt_path: str | None = None, device=DEVICE):
        self.sam_type = sam_type

        self.sam = SAM()
        self.sam.build_model(sam_type, sam_ckpt_path, device=device)
        self.gdino = GDINO()
        self.gdino.build_model(model_ckpt_path=gdino_model_ckpt_path, processor_ckpt_path=gdino_processor_ckpt_path, device=device)

    def predict(
        self,
        images_pil: list[Image.Image],
        texts_prompt: list[str],
        box_threshold: float = 0.3,
        text_threshold: float = 0.25,
    ):
        gdino_results = self.gdino.predict(images_pil, texts_prompt, box_threshold, text_threshold)
        all_results = []
        sam_images = []
        sam_boxes = []
        sam_indices = []
        for idx, result in enumerate(gdino_results):
            result = {k: (v.cpu().numpy() if hasattr(v, "numpy") else v) for k, v in result.items()}
            processed_result = {
                **result,
                "masks": [],
                "mask_scores": [],
            }

            if result["labels"]:
                sam_images.append(np.asarray(images_pil[idx]))
                sam_boxes.append(processed_result["boxes"])
                sam_indices.append(idx)

            all_results.append(processed_result)

        if sam_images:
            masks, mask_scores, _ = self.sam.predict_batch(sam_images, xyxy=sam_boxes)
            for idx, mask, score in zip(sam_indices, masks, mask_scores):
                all_results[idx].update(
                    {
                        "masks": mask,
                        "mask_scores": score,
                    }
                )
        return all_results


if __name__ == "__main__":
    model = LangSAM()
    out = model.predict(
        [Image.open("./assets/food.jpg"), Image.open("./assets/car.jpeg")],
        ["food", "car"],
    )
    print(out)
'''

with open(lang_sam_path, 'w') as f:
    f.write(new_content)

print(" Done")

In [ ]:
with open(lang_sam_path, 'r') as f:
    for line in f:
        if 'print' in line:
            print(repr(line))  # should show nothing

In [ ]:
gdino_path = "/kaggle/working/lang-segment-anything/lang_sam/models/gdino.py"

new_content = '''import torch
from PIL import Image
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

from lang_sam.models.utils import DEVICE


class GDINO:
    def build_model(self, model_ckpt_path: str | None = None, processor_ckpt_path: str | None = None, device=DEVICE):
        if not model_ckpt_path or not processor_ckpt_path:
            model_id = "IDEA-Research/grounding-dino-base"
            print(f"One or both local paths not provided. Loading from Hugging Face Hub: {model_id}")
            self.processor = AutoProcessor.from_pretrained(model_id)
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
        else:
            print(f"Attempting to load processor from local path: {processor_ckpt_path}")
            self.processor = AutoProcessor.from_pretrained(
                processor_ckpt_path,
                local_files_only=True,
                trust_remote_code=True,
            )
            print(f"Attempting to load model from local path: {model_ckpt_path}")
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
                model_ckpt_path,
                local_files_only=True,
                trust_remote_code=True,
                use_safetensors=True,
            ).to(device)

    def predict(
        self,
        images_pil: list[Image.Image],
        texts_prompt: list[str],
        box_threshold: float,
        text_threshold: float,
    ) -> list[dict]:
        # Handle single image/prompt passed directly (not in a list)
        if isinstance(images_pil, Image.Image):
            images_pil = [images_pil]
        if isinstance(texts_prompt, str):
            texts_prompt = [texts_prompt]

        results = []
        for image, prompt in zip(images_pil, texts_prompt):
            prompt = prompt if prompt[-1] == "." else prompt + "."

            inputs = self.processor(
                images=[image],
                text=[prompt],
                return_tensors="pt"
            ).to(self.model.device)

            with torch.no_grad():
                outputs = self.model(**inputs)

            result = self.processor.post_process_grounded_object_detection(
                outputs,
                inputs.input_ids,
                box_threshold,
                text_threshold=text_threshold,
                target_sizes=[image.size[::-1]],
            )
            results.extend(result)

        return results
'''

with open(gdino_path, 'w') as f:
    f.write(new_content)

print("✅ Patched successfully")

In [ ]:
gdino_path = "/kaggle/working/lang-segment-anything/lang_sam/models/gdino.py"

with open(gdino_path, 'r') as f:
    print(f.read())

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
from PIL import Image
from lang_sam import LangSAM
import os
import traceback

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/usr/local/lib/python3.12/dist-packages/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


In [2]:
input_dir=["/kaggle/input/datasets/afqwe0/clohtes/Himesh","/kaggle/input/datasets/afqwe0/clohtes/Mahesh","/kaggle/input/datasets/afqwe0/clohtes/Pratik","/kaggle/input/datasets/afqwe0/clohtes/Sworup"]
label_dir="/kaggle/working/labels"
os.makedirs(label_dir,exist_ok=True)

In [3]:
model=LangSAM()

One or both local paths not provided. Loading from Hugging Face Hub: IDEA-Research/grounding-dino-base


The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

In [4]:
folder_mapping = {
    # Casual Tops
    "T-Shirt": ("t-shirt worn by person", 0),
    "Polo": ("polo shirt on person", 23),
    "cowl neck top": ("cowl neck top", 21),
    
    # Warm Tops & Outerwear
    "Hoodie": ("hoodie sweatshirt with hood", 3),
    "Sweat Shirt": ("crewneck sweatshirt", 18),
    "Sweater": ("knit sweater pullover", 24),
    "turtle neck": ("turtleneck sweater", 22),
    "cardigan": ("cardigan open front", 20),
    "Jacket": ("jacket outerwear", 8),
    "Blazer": ("blazer suit jacket", 2),
    
    # Bottoms
    "Trousers": ("formal trousers pants", 1),
    "Jeans Pants": ("denim jeans pants", 16),
    "Baggy Pants": ("baggy wide leg pants", 15),
    "denim shorts (jorts)": ("denim jean shorts", 6),
    "Dolphin shorts": ("dolphin running shorts", 5),
    
    # Dresses & Skirts
    "skirt": ("flared skirt", 9),
    "One Piece": ("one piece dress", 17),
    "fit-and-flare dress": ("fit and flare dress", 7),
    "Sari": ("sari draped dress", 14),
    
    # Suits & Sets
    "men_suits_western_coatpant": ("mens formal suit", 10),
    "Formal Pants Shirt Set": ("formal dress shirt and pants", 12),
    
    # Footwear
    "Sneakers": ("sneakers sports shoes", 4),
    "Loafers": ("loafers slip on shoes", 11),
    
    # Headwear
    "Caps": ("baseball cap hat", 19),
    "Dhaka Topi": ("dhaka topi nepali cap", 13)
}

In [5]:
def resize_with_padding(img, target_size=640):
    """Resize keeping aspect ratio, pad to square."""
    orig_w, orig_h = img.size
    scale = target_size / max(orig_w, orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    
    img_resized = img.resize((new_w, new_h), Image.LANCZOS)
    
    # Pad to square
    padded = Image.new("RGB", (target_size, target_size), (114, 114, 114))
    pad_x = (target_size - new_w) // 2
    pad_y = (target_size - new_h) // 2
    padded.paste(img_resized, (pad_x, pad_y))
    
    return padded, scale, pad_x, pad_y

In [6]:
import shutil
import os

# Delete and recreate your labels folder
shutil.rmtree("/kaggle/working/labels")
os.makedirs(label_dir, exist_ok=True)

print(f" Cleared and recreated: {label_dir}")

 Cleared and recreated: /kaggle/working/labels


In [7]:
for folders in input_dir:
    dirs_in = os.listdir(folders)
    for img_dir in dirs_in:
        folder_path = os.path.join(folders, img_dir)

        if img_dir not in folder_mapping:
            print(f"[SKIP] No mapping for {img_dir}")
            continue

        prompt, label_id = folder_mapping[img_dir]
        print(f"Working in dir {folder_path}")

        MIN_CONFIDENCE = 0.30
        MIN_AREA_RATIO = 0.03
        MAX_AREA_RATIO = 0.98

        for img_names in os.listdir(folder_path):
            if not img_names.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(folder_path, img_names)
            img_pil = Image.open(img_path).convert("RGB")
            img_width, img_height = img_pil.size

            img_resized, scale, pad_x, pad_y = resize_with_padding(img_pil, 640)

            try:
               
                results = model.predict([img_resized], [prompt])
            except RuntimeError:
                traceback.print_exc()
                print(f"  [SKIP] {img_names}")
                continue

            result = results[0]
            boxes = result["boxes"]    # np.ndarray
            scores = result["scores"]  # np.ndarray (was logits)

            base_name = os.path.splitext(img_names)[0]
            label_path = os.path.join(label_dir, f"{base_name}.txt")

            with open(label_path, 'w') as f:
                for box, score in zip(boxes, scores):
                    # Confidence filter
                    if score < MIN_CONFIDENCE:
                        continue

                    x_min, y_min, x_max, y_max = box.tolist()

                    # Remove padding, unscale to original pixel space
                    x_min = (x_min - pad_x) / scale
                    x_max = (x_max - pad_x) / scale
                    y_min = (y_min - pad_y) / scale
                    y_max = (y_max - pad_y) / scale

                    # Area filter
                    box_area = (x_max - x_min) * (y_max - y_min)
                    area_ratio = box_area / (img_width * img_height)
                    if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
                        continue

                    # YOLO normalization
                    x_center = (x_min + x_max) / (2.0 * img_width)
                    y_center = (y_min + y_max) / (2.0 * img_height)
                    w = (x_max - x_min) / img_width
                    h = (y_max - y_min) / img_height

                    x_center = max(0.0, min(1.0, x_center))
                    y_center = max(0.0, min(1.0, y_center))
                    w = max(0.0, min(1.0, w))
                    h = max(0.0, min(1.0, h))

                    f.write(f"{label_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/fit-and-flare dress


/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/denim shorts (jorts)
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/cowl neck top
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/turtle neck
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/cardigan
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/Dolphin shorts
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/skirt
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Himesh/Jacket
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Mahesh/Dhaka Topi
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Mahesh/Sari
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Mahesh/men_suits_western_coatpant
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Mahesh/Formal Pants Shirt Set
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Mahesh/Loafers
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Pratik/Hoodie
Working in dir /kaggle/input/datasets/afqwe0/clohtes/Pr

In [8]:
shutil.make_archive('/kaggle/working/labels', 'zip', label_dir)

'/kaggle/working/labels.zip'